In [2]:
import random

import pandas as pd
import numpy as np

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import GCNConv, GAE, BatchNorm
from torch_geometric.utils import dropout_adj, from_scipy_sparse_matrix
from torch_geometric.data import Data
from torch.utils.data import DataLoader, TensorDataset
from torch.nn import BatchNorm1d

from sklearn.preprocessing import MinMaxScaler
from sklearn.decomposition import NMF
from sklearn.metrics import roc_auc_score, average_precision_score, precision_score, recall_score, f1_score

import scipy.sparse as sp

import networkx as nx
import matplotlib.pyplot as plt

from tqdm import tqdm


import torch
import torch.nn.functional as F
import numpy as np
import scipy.sparse as sp
import pandas as pd

#from torch_geometric.nn import GCNConv, GAE, BatchNorm1d
from torch_geometric.utils import dropout_adj, from_scipy_sparse_matrix
from torch_geometric.data import Data



In [3]:
# Load the data
data = pd.read_csv("/Users/nirwantandukar/input_data/Database.DREAM4/NORMALIZED_EXP/normalized_weight_matrix_all_GRN_100_01_0.2filter.csv", index_col=0)
#data = pd.read_csv("/Users/nirwantandukar/input_data/DREAM5/normalized_weight_matrix_DREAM5_net1_0.2filter.csv", index_col=0)
print(f"Loaded data with shape: {data.shape}")

# Get the list of algorithms
columns = data.columns
algorithms = sorted(set(col.split('.')[0] for col in columns))
print(f"Algorithms found: {algorithms}")

num_genes = data.shape[0]
gene_names = data.index.tolist()
print(f"Number of genes: {num_genes}")

# Node features (identity matrix)
node_features = np.identity(num_genes)
print(f"Node features shape: {node_features.shape}")

Loaded data with shape: (100, 1700)
Algorithms found: ['GRN_ARACNE', 'GRN_BC3NET', 'GRN_C3NET', 'GRN_CLR', 'GRN_GENIE3_ET', 'GRN_GENIE3_RF', 'GRN_GRENIST', 'GRN_GRNBOOST2', 'GRN_INFERELATOR', 'GRN_KBOOST', 'GRN_LEAP', 'GRN_MEOMI', 'GRN_MRNET', 'GRN_MRNETB', 'GRN_NONLINEARODES', 'GRN_PCIT', 'GRN_TIGRESS']
Number of genes: 100
Node features shape: (100, 100)


In [4]:
# ---------------------------
# Helper: Normalized Laplacian
# ---------------------------
def normalize_adj(adj):
    """Compute the normalized Laplacian of an adjacency matrix."""
    adj = sp.coo_matrix(adj)
    rowsum = np.array(adj.sum(1)).flatten()
    # Add a small constant for numerical stability
    d_inv_sqrt = np.power(rowsum + 1e-5, -0.5)
    d_inv_sqrt[np.isinf(d_inv_sqrt)] = 0.0
    d_mat_inv_sqrt = sp.diags(d_inv_sqrt)
    identity = sp.eye(adj.shape[0])
    laplacian = identity - d_mat_inv_sqrt.dot(adj).dot(d_mat_inv_sqrt)
    return laplacian.tocoo()

# ---------------------------
# GCN Encoder (shared across views)
# ---------------------------
class GCNEncoder(torch.nn.Module):
    def __init__(self, in_channels, hidden_channels, out_channels, num_layers=4, 
                 dropout_rate=0.01, edge_dropout_rate=0.01):
        super(GCNEncoder, self).__init__()
        self.num_layers = num_layers
        self.dropout_rate = dropout_rate
        self.edge_dropout_rate = edge_dropout_rate
        
        # First layer: input to 2x hidden size
        self.convs = torch.nn.ModuleList()
        self.convs.append(GCNConv(in_channels, 2 * hidden_channels, cached=False))
        
        # Add intermediate layers with the same dimensions
        for _ in range(num_layers - 2):
            self.convs.append(GCNConv(2 * hidden_channels, 2 * hidden_channels, cached=False))
        
        # Final layer: hidden size to output embedding size
        self.convs.append(GCNConv(2 * hidden_channels, out_channels, cached=False))
    
        # Batch normalization layers for each intermediate layer
        self.bns = torch.nn.ModuleList([BatchNorm1d(2 * hidden_channels, momentum=0.1) 
                                        for _ in range(num_layers - 1)])
    
        self.dropout = torch.nn.Dropout(p=dropout_rate)
        self.activation_hidden = torch.nn.ReLU  # for hidden layers
        self.activation_output = torch.nn.Sigmoid()  # for final layer (optional)

    def forward(self, x, edge_index):
        # Apply edge dropout to the connectivity
        edge_index, _ = dropout_adj(edge_index, p=self.edge_dropout_rate, training=self.training)
        
        # First layer (without batch norm)
        x = self.convs[0](x, edge_index)
        x = F.relu(x)
    
        # Intermediate layers: conv -> batch norm -> relu -> dropout -> residual
        for i in range(1, self.num_layers - 1):
            x_residual = x
            x = self.convs[i](x, edge_index)
            x = self.bns[i-1](x)
            x = F.relu(x)
            x = self.dropout(x)
            x = x + x_residual  # residual connection
            
        # Final layer (no batch norm or dropout)
        x = self.convs[-1](x, edge_index)
        x = F.relu(x)
        return x

# ---------------------------
# Joint Multi–View GAE Training (Shared Encoder)
# ---------------------------
def apply_joint_gae_to_adjacency(data, algorithms, node_features, embedding_dim=16, 
                                 epochs=200, dropout_rate=0.1, edge_dropout_rate=0.01, weight_decay=1e-8):
    """
    Jointly train a GAE on multiple views (algorithms) with shared node features.
    
    Parameters:
        data (pd.DataFrame): The full input DataFrame containing 100×1700 values 
                             (each algorithm’s 100×100 matrix is stored as columns).
        algorithms (list): List of algorithm prefixes (e.g., ['GRN_PCIT', ...]).
        node_features (np.array): Shared node features (typically identity, shape: 100×100).
        embedding_dim (int): Dimension of the latent space.
        epochs (int): Number of training epochs.
        dropout_rate (float): Dropout probability for the encoder.
        edge_dropout_rate (float): Edge dropout probability.
        weight_decay (float): Weight decay for the optimizer.
        
    Returns:
        reconstructed (dict): Maps algorithm name to its reconstructed (normalized) adjacency matrix.
    """
    num_genes = node_features.shape[0]
    # Preprocess each algorithm’s graph and store the edge indices
    graph_data = {}
    for alg in algorithms:
        alg_columns = [col for col in data.columns if col.startswith(alg + '.')]
        if len(alg_columns) != num_genes:
            print(f"⚠️ Skipping {alg}: expected {num_genes} columns, got {len(alg_columns)}")
            continue
        alg_matrix = data[alg_columns].values.reshape((num_genes, num_genes))
        # Normalize the adjacency matrix: add identity then normalize using the Laplacian approach
        adj_sparse = sp.csr_matrix(alg_matrix)
        adj_normalized = normalize_adj(adj_sparse + sp.eye(adj_sparse.shape[0]))
        # Use torch_geometric utility to get edge_index from the normalized sparse matrix
        edge_index, _ = from_scipy_sparse_matrix(adj_normalized)
        graph_data[alg] = {
            "edge_index": edge_index,
            # You might want to store the raw matrix if needed for further processing
            "raw_adj": alg_matrix
        }
    
    # Create a tensor for node features (shared across all graphs)
    x = torch.tensor(node_features, dtype=torch.float)
    
    # Instantiate the GAE model (using the shared encoder)
    encoder = GCNEncoder(in_channels=x.shape[1], hidden_channels=embedding_dim, 
                           out_channels=embedding_dim, dropout_rate=dropout_rate, 
                           edge_dropout_rate=edge_dropout_rate)
    model = GAE(encoder)
    
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model = model.to(device)
    x = x.to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=0.01, weight_decay=weight_decay)
    
    model.train()
    for epoch in range(1, epochs + 1):
        optimizer.zero_grad()
        total_loss = 0.0
        # Loop over each algorithm/view
        for alg, gdata in graph_data.items():
            edge_index = gdata["edge_index"].to(device)
            # Each view produces its own latent representation using the shared encoder
            z = model.encode(x, edge_index)
            loss = model.recon_loss(z, edge_index)
            total_loss += loss
        total_loss.backward()
        optimizer.step()
    
    # After training, for each view compute its reconstructed adjacency matrix
    model.eval()
    reconstructed = {}
    with torch.no_grad():
        for alg, gdata in graph_data.items():
            edge_index = gdata["edge_index"].to(device)
            z = model.encode(x, edge_index)
            adj_reconstructed = model.decoder.forward_all(z).cpu().numpy()
            # Normalize reconstruction to [0, 1]
            adj_min = adj_reconstructed.min()
            adj_max = adj_reconstructed.max()
            normalized_adj = (adj_reconstructed - adj_min) / (adj_max - adj_min)
            reconstructed[alg] = normalized_adj
    return reconstructed

# ---------------------------
# Running Joint Multi–View GAE with Filtering
# ---------------------------
def run_joint_gae_with_algo_filtering(data, algorithms, node_features, thresholds, 
                                      num_iterations=2, algo_threshold=2,
                                      embedding_dim=16, epochs=200, dropout_rate=0.1, 
                                      edge_dropout_rate=0.01, weight_decay=1e-8):
    """
    Run the joint (multi–view) GAE training repeatedly and filter the results across iterations.
    
    For each iteration, the joint GAE is trained on all algorithms/views simultaneously,
    then for each algorithm the reconstructed matrix is binarized using the threshold.
    Finally, only edges that appear in at least `algo_threshold` iterations are retained.
    
    Parameters:
        data (pd.DataFrame): The input DataFrame (100×1700).
        algorithms (list): List of algorithm prefixes.
        node_features (np.array): Shared node features (100×100).
        thresholds (list): List of binarization thresholds (e.g., [0, 0.4, ...]).
        num_iterations (int): Number of GAE runs.
        algo_threshold (int): Minimum number of iterations an edge must be present.
        embedding_dim, epochs, dropout_rate, edge_dropout_rate, weight_decay: Hyperparameters.
        
    Returns:
        filtered_results (dict): Nested dict: threshold → algorithm → filtered (binary) 100×100 matrix.
    """
    num_genes = data.shape[0]
    filtered_results = {}

    for threshold in thresholds:
        print(f"\n🔹 Running Joint GAE with binarization threshold: {threshold}")
        # Prepare storage for binary reconstructed matrices per algorithm over iterations
        filtered_results[f'threshold_{threshold}'] = {alg: [] for alg in algorithms}

        for iteration in range(num_iterations):
            print(f"  ⏳ Iteration {iteration + 1}/{num_iterations}")
            reconstructed_dict = apply_joint_gae_to_adjacency(
                data=data,
                algorithms=algorithms,
                node_features=node_features,
                embedding_dim=embedding_dim,
                epochs=epochs,
                dropout_rate=dropout_rate,
                edge_dropout_rate=edge_dropout_rate,
                weight_decay=weight_decay
            )
            for alg in algorithms:
                rec = reconstructed_dict.get(alg)
                if rec is not None:
                    # Binarize the reconstructed matrix using the threshold
                    binary = (rec > threshold).astype(int)
                    filtered_results[f'threshold_{threshold}'][alg].append(binary)

        # Filter edges across iterations: keep only edges seen in ≥ algo_threshold iterations
        for alg in algorithms:
            matrices = np.stack(filtered_results[f'threshold_{threshold}'][alg], axis=0)
            filtered = (np.sum(matrices, axis=0) >= algo_threshold).astype(int)
            filtered_results[f'threshold_{threshold}'][alg] = filtered

    return filtered_results



In [11]:
# ---------------------------
# Example usage:
# ---------------------------
# Define parameters for filtering
thresholds = [0]         # Example threshold(s)
num_iterations = 10      # Number of runs per threshold
algo_threshold = 5       # Minimum iterations an edge must appear in

# Run joint multi–view GAE with filtering
filtered_results = run_joint_gae_with_algo_filtering(
    data=data,
    algorithms=algorithms,
    node_features=node_features,
    thresholds=thresholds,
    num_iterations=num_iterations,
    algo_threshold=algo_threshold,
    embedding_dim=16,
    epochs=200,
    dropout_rate=0.01,
    edge_dropout_rate=0.01,
    weight_decay=1e-8
)



🔹 Running Joint GAE with binarization threshold: 0
  ⏳ Iteration 1/10


/opt/anaconda3/envs/machine_learning/lib/python3.12/site-packages/torch_geometric/deprecation.py:26: UserWarning: 'dropout_adj' is deprecated, use 'dropout_edge' instead
  warnings.warn(out)


  ⏳ Iteration 2/10


/opt/anaconda3/envs/machine_learning/lib/python3.12/site-packages/torch_geometric/deprecation.py:26: UserWarning: 'dropout_adj' is deprecated, use 'dropout_edge' instead
  warnings.warn(out)


  ⏳ Iteration 3/10


/opt/anaconda3/envs/machine_learning/lib/python3.12/site-packages/torch_geometric/deprecation.py:26: UserWarning: 'dropout_adj' is deprecated, use 'dropout_edge' instead
  warnings.warn(out)


... (10 outputs left out)

  ⏳ Iteration 9/10


/opt/anaconda3/envs/machine_learning/lib/python3.12/site-packages/torch_geometric/deprecation.py:26: UserWarning: 'dropout_adj' is deprecated, use 'dropout_edge' instead
  warnings.warn(out)


  ⏳ Iteration 10/10


/opt/anaconda3/envs/machine_learning/lib/python3.12/site-packages/torch_geometric/deprecation.py:26: UserWarning: 'dropout_adj' is deprecated, use 'dropout_edge' instead
  warnings.warn(out)


In [12]:
filtered_results

{'threshold_0': {'GRN_ARACNE': array([[1, 1, 0, ..., 0, 0, 0],
         [1, 1, 0, ..., 0, 0, 0],
         [0, 0, 0, ..., 0, 0, 0],
         ...,
         [0, 0, 0, ..., 1, 0, 0],
         [0, 0, 0, ..., 0, 1, 1],
         [0, 0, 0, ..., 0, 1, 1]]),
  'GRN_BC3NET': array([[1, 0, 0, ..., 0, 0, 0],
         [0, 1, 0, ..., 0, 0, 0],
         [0, 0, 1, ..., 1, 0, 0],
         ...,
         [0, 0, 1, ..., 1, 0, 0],
         [0, 0, 0, ..., 0, 1, 1],
         [0, 0, 0, ..., 0, 1, 1]]),
  'GRN_C3NET': array([[1, 1, 0, ..., 0, 0, 0],
         [1, 1, 0, ..., 0, 0, 0],
         [0, 0, 0, ..., 0, 0, 0],
         ...,
         [0, 0, 0, ..., 1, 0, 0],
         [0, 0, 0, ..., 0, 1, 0],
         [0, 0, 0, ..., 0, 0, 1]]),
  'GRN_CLR': array([[1, 1, 1, ..., 1, 1, 1],
         [1, 1, 1, ..., 1, 1, 1],
         [1, 1, 1, ..., 1, 1, 1],
         ...,
         [1, 1, 1, ..., 1, 1, 1],
         [1, 1, 1, ..., 1, 1, 1],
         [1, 1, 1, ..., 1, 1, 1]]),
  'GRN_GENIE3_ET': array([[1, 1, 1, ..., 1, 1, 1],
  

In [13]:
# Applying NMF to 17 algorithm
def fuse_with_joint_nmf_from_filtered(filtered_results, threshold, rank=20, num_genes=100):
    """
    Fuses multiple filtered binary adjacency matrices using joint NMF.
    Each matrix must be (num_genes x num_genes), one per algorithm.
    """
    all_flattened = []

    for alg, matrix in filtered_results[f'threshold_{threshold}'].items():
        assert matrix.shape == (num_genes, num_genes), f"{alg} shape is invalid: {matrix.shape}"
        flattened = matrix.flatten()
        all_flattened.append(flattened)

    # all_flattened = 17 x 10000 → each row is one algorithm's flattened adjacency matrix
    nmf_input = np.array(all_flattened)  # shape: (17, 10000)
    print(f"Matrix shape (before transpose): {nmf_input.shape}")

    V = nmf_input.T  # (10000, 17)
    print(f"NMF input shape: {V.shape}")

    nmf_model = NMF(n_components=rank, init='random', random_state=42, max_iter=1000)
    W = nmf_model.fit_transform(V)
    H = nmf_model.components_

    # Reconstruct the fused graph
    V_reconstructed = np.dot(W, H).mean(axis=1)  # shape: (10000,)

    # RESHAPE using known num_genes = 100, not inferred!
    fused_adj = V_reconstructed.reshape((num_genes, num_genes))

    # Normalize to [0, 1]
    fused_adj = (fused_adj - fused_adj.min()) / (fused_adj.max() - fused_adj.min())

    return fused_adj



In [14]:
# Running the NMF fusion
fused_adj_nmf = fuse_with_joint_nmf_from_filtered(filtered_results, threshold=0, rank=20)


Matrix shape (before transpose): (17, 10000)
NMF input shape: (10000, 17)


In [15]:
threshold = 0.6
fused_binary = (fused_adj_nmf >= threshold).astype(int)

In [16]:
# Function to extract nodes and edges from the binary matrix
def count_nodes_edges(final_matrix):
    # Count edges in the upper triangle (to avoid double counting for undirected graphs)
    num_edges = np.sum(np.triu(final_matrix, k=1))
    
    # Count nodes (genes with at least one edge, i.e., rows/columns with non-zero values)
    connected_nodes = np.sum(np.any(final_matrix, axis=1))  # Count rows with non-zero values
    num_nodes = connected_nodes
    
    return num_nodes, num_edges

# Function to extract gene interactions (edges) from a binarized matrix
def extract_gene_interactions(final_matrix, gene_names):
    edges = []
    num_genes = final_matrix.shape[0]

    # Loop through the upper triangle of the matrix to avoid duplicates (since it's undirected)
    for i in range(num_genes):
        for j in range(i + 1, num_genes):
            if final_matrix[i, j] == 1:  # Check if there's an interaction
                edges.append(f"{gene_names[i]}-{gene_names[j]}")
    
    return edges


In [17]:
# Count nodes and edges
num_nodes, num_edges = count_nodes_edges(fused_binary)
print(f"Nodes: {num_nodes}, Edges: {num_edges}")

# Get the edge list
nmf_edge_list = extract_gene_interactions(fused_binary, gene_names)
print(f"Sample edges: {nmf_edge_list[:5]}")

Nodes: 96, Edges: 372
Sample edges: ['G1-G16', 'G1-G23', 'G1-G29', 'G1-G36', 'G1-G37']


In [18]:
# Convert ground truth matrix to edge list
def convert_matrix_to_edge_list(matrix):
    edges = []
    for source in matrix.index:
        for target in matrix.columns:
            if matrix.at[source, target] == 1:
                edges.append((source, target))
    return edges

# Load known (ground truth) interactions
file_path = "/Users/nirwantandukar/input_data/Database.DREAM4/GS/dream4_100_01_gs.csv"
ground_truth = pd.read_csv(file_path, index_col=0)
print(f"Ground truth loaded with shape: {ground_truth.shape}")

# Get the ground truth edges
known_edges_set = set(convert_matrix_to_edge_list(ground_truth))
print(f"Number of known edges: {len(known_edges_set)}")


Ground truth loaded with shape: (100, 100)
Number of known edges: 175


In [19]:
def convert_str_edges_to_tuples(edge_list):
    edge_tuples = set()
    for edge in edge_list:
        a, b = edge.split('-')
        # Add both (a, b) and (b, a) to ensure directionality doesn't affect matching
        edge_tuples.add((a, b))
        edge_tuples.add((b, a))
    return edge_tuples

predicted_edge_set = convert_str_edges_to_tuples(nmf_edge_list)

def symmetrize_edges(edge_set):
    symmetric_set = set()
    for a, b in edge_set:
        symmetric_set.add((a, b))
        symmetric_set.add((b, a))
    return symmetric_set

known_edges_symmetric = symmetrize_edges(known_edges_set)



In [20]:
# True Positives (correct predictions)
true_positives = predicted_edge_set.intersection(known_edges_symmetric)

# False Positives (predicted but not known)
false_positives = predicted_edge_set - known_edges_symmetric

# False Negatives (missed known edges)
false_negatives = known_edges_symmetric - predicted_edge_set

# Precision, Recall, F1
tp = len(true_positives) // 2  # divide by 2 because of symmetric edges
fp = len(false_positives) // 2
fn = len(false_negatives) // 2

precision = tp / (tp + fp) if (tp + fp) > 0 else 0
recall = tp / (tp + fn) if (tp + fn) > 0 else 0
f1 = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0

print(f"TP: {tp}, FP: {fp}, FN: {fn}")
print(f"Precision: {precision:.4f}, Recall: {recall:.4f}, F1 Score: {f1:.4f}")


TP: 41, FP: 331, FN: 127
Precision: 0.1102, Recall: 0.2440, F1 Score: 0.1519


In [21]:
#### AUPR and AUROC

from sklearn.metrics import roc_auc_score, average_precision_score

def compute_auroc_aupr(predicted_adj, ground_truth_df, gene_names):
    """
    Computes AUROC and AUPR given predicted adjacency (scores) and ground truth (binary).
    Assumes both are 100x100 matrices with matching gene order.
    """

    # Convert ground truth DataFrame to a numpy binary matrix
    gt_matrix = ground_truth_df.loc[gene_names, gene_names].values.astype(int)

    # Flatten only upper triangle (excluding diagonal)
    triu_indices = np.triu_indices_from(gt_matrix, k=1)
    gt_flat = gt_matrix[triu_indices]
    pred_flat = predicted_adj[triu_indices]

    # AUROC and AUPR
    auroc = roc_auc_score(gt_flat, pred_flat)
    aupr = average_precision_score(gt_flat, pred_flat)

    print(f"AUROC: {auroc:.4f}")
    print(f"AUPR : {aupr:.4f}")
    
    return auroc, aupr


In [22]:
# Call the scorer
compute_auroc_aupr(fused_adj_nmf, ground_truth, gene_names)

AUROC: 0.7494
AUPR : 0.0629


(0.7493922007196343, 0.06289304216053271)

In [42]:
import numpy as np

# Assume your filtered_results dictionary from your previous step:
# filtered_results['threshold_0'][algorithm] -> binary matrix (100×100)

algorithms = filtered_results['threshold_0'].keys()
consensus_matrix = np.zeros((100, 100))

# Sum across algorithms:
for alg in algorithms:
    consensus_matrix += filtered_results['threshold_0'][alg]

# Apply threshold (e.g., edges appearing in at least 5 algorithms):
threshold =8
final_consensus_graph = (consensus_matrix >= threshold).astype(int)

print(final_consensus_graph)


ValueError: operands could not be broadcast together with shapes (100,100) (1643,1643) (100,100) 

In [44]:
# Count nodes and edges
num_nodes, num_edges = count_nodes_edges(final_consensus_graph)
print(f"Nodes: {num_nodes}, Edges: {num_edges}")

# Get the edge list
nmf_edge_list = extract_gene_interactions(final_consensus_graph, gene_names)
print(f"Sample edges: {nmf_edge_list[:5]}")

NameError: name 'final_consensus_graph' is not defined

In [46]:
# True Positives (correct predictions)
true_positives = predicted_edge_set.intersection(known_edges_symmetric)

# False Positives (predicted but not known)
false_positives = predicted_edge_set - known_edges_symmetric

# False Negatives (missed known edges)
false_negatives = known_edges_symmetric - predicted_edge_set

# Precision, Recall, F1
tp = len(true_positives) // 2  # divide by 2 because of symmetric edges
fp = len(false_positives) // 2
fn = len(false_negatives) // 2

precision = tp / (tp + fp) if (tp + fp) > 0 else 0
recall = tp / (tp + fn) if (tp + fn) > 0 else 0
f1 = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0

print(f"TP: {tp}, FP: {fp}, FN: {fn}")
print(f"Precision: {precision:.4f}, Recall: {recall:.4f}, F1 Score: {f1:.4f}")


TP: 19, FP: 76280, FN: 149
Precision: 0.0002, Recall: 0.1131, F1 Score: 0.0005


In [48]:
# Call the scorer
compute_auroc_aupr(final_consensus_graph, ground_truth, gene_names)

NameError: name 'final_consensus_graph' is not defined

In [57]:
### DREAM 5

# Load the edge list CSV
file_path = "/Users/nirwantandukar/input_data/DREAM5/net1_in_silico_gold_standard.csv"
ground_truth = pd.read_csv(file_path)

# Confirm expected columns are there
print("Columns:", ground_truth.columns)

# Convert edge list to set of tuples
known_edges_set = set(zip(ground_truth['from'], ground_truth['to']))
print(f"Loaded {len(known_edges_set)} known edges from edge list.")
known_edges_set

# DREAM 5 


from sklearn.metrics import precision_recall_curve, roc_curve, auc
import numpy as np
import pandas as pd

def compute_auroc_aupr(predicted_adj, ground_truth_df, gene_names):
    """
    Computes AUROC and AUPR from:
    - predicted_adj: N×N numpy array of scores
    - ground_truth_df: edge list with columns ['from', 'to']
    - gene_names: list of gene names matching predicted_adj
    """

    # Build a ground truth binary matrix from edge list
    N = len(gene_names)
    gene_to_index = {gene: i for i, gene in enumerate(gene_names)}
    gt_binary = np.zeros((N, N), dtype=int)

    for _, row in ground_truth_df.iterrows():
        src, tgt = row['from'], row['to']
        if src in gene_to_index and tgt in gene_to_index:
            i, j = gene_to_index[src], gene_to_index[tgt]
            gt_binary[i, j] = 1

    # Flatten upper triangle only (no self loops)
    triu = np.triu_indices(N, k=1)
    y_true = gt_binary[triu]
    y_scores = predicted_adj[triu]

    # Compute metrics
    fpr, tpr, _ = roc_curve(y_true, y_scores)
    precision, recall, _ = precision_recall_curve(y_true, y_scores)

    auroc = auc(fpr, tpr)
    aupr = auc(recall, precision)

    print(f"AUROC: {auroc:.4f}")
    print(f"AUPR:  {aupr:.4f}")

    return auroc, aupr


import numpy as np

def edge_list_to_binary_matrix(edge_list, gene_names):
    N = len(gene_names)
    gene_to_index = {gene: i for i, gene in enumerate(gene_names)}
    matrix = np.zeros((N, N), dtype=int)

    for edge in edge_list:
        try:
            src, tgt = edge.split('-')
            if src in gene_to_index and tgt in gene_to_index:
                i, j = gene_to_index[src], gene_to_index[tgt]
                matrix[i, j] = 1
        except ValueError:
            continue  # skip malformed lines

    return matrix

# Convert edge list into binary predicted matrix
predicted_adj = edge_list_to_binary_matrix(nmf_edge_list, gene_names)

# Run scorer using your ground truth edge list
compute_auroc_aupr(predicted_adj, ground_truth, gene_names)



Columns: Index(['from', 'to', 'weight'], dtype='object')
Loaded 4012 known edges from edge list.
AUROC: 0.5771
AUPR:  0.1091


(0.5770904537244453, 0.10910093667123284)